In [12]:
import nest_asyncio 
nest_asyncio.apply()

import asyncio
import os
from pathlib import Path
from ollama import AsyncClient


In [1]:
MAX_CHARS_PER_CHUNK = 8000 

# 1. System Prompt targeting Qwen Coder for file analysis
SYSTEM_PROMPT = """
Role: Senior Java/Groovy & Grails Monolith Systems Architect
Task: Analyze a hybrid Grails 3.x and Spring Boot monolithic application for structural layout, caching architecture, data flow bottlenecks, and Redis integration.

[CONTEXT OVERVIEW]
The target application is a monolithic system built using Grails 3.x (operating on top of a Spring Boot container). It uses GORM for relational persistence alongside Redis for caching or transient data structures.

[INSTRUCTIONS FOR ANALYSIS EXECUTIONS]
Parse the code through the lens of a Grails 3.x + Spring Boot structural hierarchy.
NOTE: If this is a chunked file, analyze it as a partial snippet of a larger class context.

[OUTPUT FORMAT REQUIREMENT]
Output your assessment exactly in this structured format for the code provided:

### 📄 File: {filepath_placeholder}
#### 🏛️ Core Artifact Mapping
- **Component Path:** [Path]
- **Component Type:** [Grails Service / Spring Component / GORM Domain / Controller]
- **Lifecycle Management:** [Grails Lifecycle Managed / Spring Boot Container Managed]

#### 🔄 Dependencies & Data Flow
- **Inbound Injections:** [What beans/services are injected into this class]
- **Outbound Operations:** [What databases, services, or Redis keys does this class touch]

#### ⚡ Caching & Redis Footprint
- **Caching Mechanism:** [None / @Cacheable / Direct RedisTemplate / GORM Redis]
- **Key Strategy:** [Provide the Redis key pattern used, e.g., 'user:profile:status:\\$id']
- **Potential Failure Modes:** [List serialization errors, connection leaks, or GORM lazy-load issues]

---
"""


In [14]:
# 2. Aggregator Prompt for final summary generation
SUMMARY_PROMPT = """
Role: Data Systems Auditor
Task: Analyze the provided code evaluation text and generate a structured aggregate catalog summarizing all Redis cache configurations.

Instructions:
Extract all identified Redis cache keys, prefixes, data types, namespaces, and vulnerabilities from the text.

Output exactly in this format at the beginning of the file:
## 📊 Enterprise Redis Cache Catalog Summary
### 🔑 Identified Keys & Namespaces

| Namespace / Prefix | Estimated Type | Originating Component | Eviction Trigger |
| :--- | :--- | :--- | :--- |
| [e.g., `user:session:`] | [String/Hash] | [AuthService] | [Logout / Session Timeout] |

### ⚡ Caching Vulnerabilities & Failure Risks
- [List critical architectural issues like cross-container serialization gaps, un-annotated transactional boundaries, or tight GORM lazy-loading hooks here]
"""


In [15]:
# Config targets & limits
TARGET_EXTENSIONS = {'.groovy', '.java', '.yml', '.properties'}
EXCLUDE_DIRS = {'build', '.gradle', '.git', 'node_modules', '.gsp'}
MAX_CHARS_PER_CHUNK = 6000  


In [16]:
def split_code_file(code_content: str, max_chars: int = MAX_CHARS_PER_CHUNK) -> list[str]:
    """Syntax-aware file-splitter component separating blocks at top-level brace boundaries."""
    lines = code_content.splitlines()
    chunks = []
    current_chunk = []
    current_length = 0
    brace_count = 0
    
    for line in lines:
        stripped = line.strip()
        current_chunk.append(line)
        current_length += len(line) + 1
        
        brace_count += stripped.count('{')
        brace_count -= stripped.count('}')
        
        if current_length >= max_chars and brace_count <= 1:
            chunks.append("\n".join(current_chunk))
            current_chunk = []
            current_length = 0
            
    if current_chunk:
        chunks.append("\n".join(current_chunk))
        
    return chunks

async def analyze_file(client: AsyncClient, model_name: str, file_path: Path, output_file: Path):
    """Reads, chunks, and streams code analysis from Qwen Coder directly into a report file."""
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            code_content = f.read()
        
        if not code_content.strip():
            return

        code_chunks = split_code_file(code_content)
        total_chunks = len(code_chunks)
        
        try:
            relative_path = file_path.relative_to(Path.cwd())
        except ValueError:
            relative_path = file_path.name

        for index, chunk in enumerate(code_chunks, start=1):
            print(f"⏳ Processing: {relative_path} [{index}/{total_chunks}]...")

            with open(output_file, 'a', encoding='utf-8') as out_f:
                out_f.write(f"\n## Analysis for `{relative_path}` (Part {index}/{total_chunks})\n")

            user_message = (
                f"Project File: {file_path}\n"
                f"Chunk Status: Part {index} of {total_chunks}\n\n"
                f"Code Snippet:\n```\n{chunk}\n```"
            )

            response_stream = await client.chat(
                model=model_name,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': user_message}
                ],
                stream=True
            )

            async for chunk_response in response_stream:
                content = chunk_response['message']['content']
               # with open(output_file, 'a', encoding='utf-8') as out_f:
                out_f.write(content)
            
            # with open(output_file, 'a', encoding='utf-8') as out_f:
            out_f.write("\n\n---\n")
            
            if total_chunks > 1:
                await asyncio.sleep(0.2)

    except Exception as e:
        print(f"❌ Error analyzing {file_path}: {e}")

In [11]:

async def generate_redis_summary(client: AsyncClient, model_name: str, output_file: Path):
    """Reads the raw compiled report, requests an aggregated Redis summary block, and prepends it."""
    print("📊 Compiling Redis Cache Catalog summary block...")
    try:
        with open(output_file, 'r', encoding='utf-8') as f:
            compiled_report_data = f.read()

        response = await client.chat(
            model=model_name,
            messages=[
                {'role': 'system', 'content': SUMMARY_PROMPT},
                {'role': 'user', 'content': f"Generate the catalog based on this analysis text:\n\n{compiled_report_data}"}
            ]
        )
        
        summary_content = response['message']['content']

        # Prepend the generated summary to the top of the file layout
        with open(output_file, 'r', encoding='utf-8') as f:
            original_content = f.read()

        with open(output_file, 'w', encoding='utf-8') as f:
            f.write("# Monolithic Application Code Structure Report\n")
            f.write(f"**Framework Matrix:** Grails 3.x / Spring Boot / GORM / Redis\n\n")
            f.write(summary_content)
            f.write("\n\n# Detailed File Breakdown\n")
            f.write(original_content.replace("# Monolithic Application Code Structure Report\n**Framework Matrix:** Grails 3.x / Spring Boot / GORM / Redis\n\n", ""))

        print("✨ Redis Cache Catalog summary successfully generated at the top of the report.")

    except Exception as e:
        print(f"❌ Error generating summary section: {e}")

In [17]:

async def main():
    MODEL_NAME = 'qwen2.5-coder:7b' 
    EXCLUDE_DIRS = {'build', '.gradle', '.git', 'node_modules', '.gsp'}
    TARGET_EXTENSIONS = {'.groovy', '.java', '.yml', '.properties'}
    
    client = AsyncClient()
    project_root = Path.cwd()
    
    output_report = project_root / "markdown_report.md"
    with open(output_report, 'w', encoding='utf-8') as f:
        f.write("# Monolithic Application Code Structure Report\n")
        f.write(f"**Framework Matrix:** Grails 3.x / Spring Boot / GORM / Redis\n\n")

    print(f"🚀 Starting Monolith Structure Scan.")
    print(f"📝 Output pipeline targeting: {output_report.name}")
    
    files_to_analyze = [ "/Users/cherylhoekstra/projects/grails-demo/grails-app"]
    for root, dirs, files in os.walk(project_root):
        dirs[:] = [d for d in dirs if d not in EXCLUDE_DIRS]
        for file in files:
            file_path = Path(root) / file
            if file_path.suffix in TARGET_EXTENSIONS and file_path.name != "markdown_report.md":
                files_to_analyze.append(file_path)

    if not files_to_analyze:
        print("⚠️ No valid source files found matching targets.")
        return

    print(f"📋 Found {len(files_to_analyze)} files. Processing pipeline running...")

    for file_path in files_to_analyze:
        await analyze_file(client, MODEL_NAME, file_path, output_report)
        await asyncio.sleep(0.5)

    # Post-processing hook: Generate and prepend the summary
    await generate_redis_summary(client, MODEL_NAME, output_report)
    print(f"\n✨ Success! Report written to `{output_report.name}`")

if __name__ == "__main__":
    asyncio.run(main())


🚀 Starting Monolith Structure Scan.
📝 Output pipeline targeting: markdown_report.md
📋 Found 1 files. Processing pipeline running...
❌ Error analyzing /Users/cherylhoekstra/projects/grails-demo/grails-app: [Errno 21] Is a directory: '/Users/cherylhoekstra/projects/grails-demo/grails-app'
📊 Compiling Redis Cache Catalog summary block...
✨ Redis Cache Catalog summary successfully generated at the top of the report.

✨ Success! Report written to `markdown_report.md`
